In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import polars as pl
from PIL import Image
from polars import DataType
from tqdm.auto import tqdm
from wordcloud import WordCloud

In [ ]:
DATA_PATH = Path("/mnt/Fast2T/data/")

# Research Questions

## Precomputation

Stuff that we precomputed because the precomputation alone required special settings like 256GB swap (additional disk ram) to the existing 64GB.

Websites grouped by domain and all features aggregated

In [ ]:
(
    pl
    .scan_parquet(DATA_PATH / "stage_6/*.parquet")
    .select("host", "is_ai", "ai_topic", "topic")
    .group_by("host")
    .agg(
        (pl.col("is_ai") == "AI").mean().alias("ai"),
        (pl.col("is_ai") == "HUMAN").mean().alias("human"),
        (pl.col("ai_topic") == "AI").mean().alias("ai_topic"),
        pl.struct(
            [
                pl.col("topic").filter((pl.col("topic") == "NEWS") | (pl.col("topic") == "NEWSAPI")).len().alias(
                    "NEWS"
                ) / pl.len(),
                *[
                    pl.col("topic").filter(pl.col("topic") == t).len().alias(t) / pl.len()
                    for t in ["BLOG", "WIKI", "SHOP"]
                ]
            ]
        ).alias("topic"),
        pl.len().alias("samples")
    )
    .sink_parquet(DATA_PATH / "domain_stats.parquet")
)

Domain Graph

In [ ]:
(
    pl
    .scan_parquet(DATA_PATH / "stage_6/*.parquet")
    .select("host", "outflow")
    .explode("outflow")
    .drop_nulls()
    .unnest("outflow")
    .rename({ "host": "from", "url": "to" })
    .group_by("from", "to")
    .agg(pl.col("occurrences").sum())
    .sink_parquet(DATA_PATH / "domain_outflow.parquet")
)

global lemma usage (stems is a deprecated name because we first used a stemmer)

In [ ]:
(
    pl
    .scan_parquet(DATA_PATH / "stage_6/*.parquet")
    .select("stems", "is_ai", "ai_topic", "topic")
    .explode("stems")
    .unnest("stems")
    .group_by("is_ai", "ai_topic", "topic", "stem")
    .agg(pl.col("occurrence").sum())
    .with_columns(
        pl.col("occurrence") / pl.col("occurrence")
        .sum()
        .over("is_ai", "ai_topic", "topic")
        .alias("p")
    )
    .sink_parquet(DATA_PATH / "word_distribution.parquet")
)

In [ ]:
df = (
    pl
    .scan_parquet(DATA_PATH / "word_distribution.parquet")
    .group_by("is_ai", "stem")
    .agg(pl.col("occurrence").sum(), pl.col("p").sum())
    .pivot("is_ai", index="stem", values="p", on_columns=["AI", "HUMAN", "UNKNOWN"])
    .fill_null(0)
    .collect()
)

threshold = {
    col: df[col].sort(descending=True)[9999]
    for col in ["AI", "HUMAN", "UNKNOWN"]
}

(
    df
    .filter(
        (pl.col("AI") >= threshold["AI"])
        | (pl.col("HUMAN") >= threshold["HUMAN"])
        | (pl.col("UNKNOWN") >= threshold["UNKNOWN"])
    )
    .write_parquet(DATA_PATH / "word_distribution_top.parquet")
)

## Research Questions

### Word Usage of AI and Non-AI Content

In [ ]:
df = (
    pl
    .read_parquet(DATA_PATH / "word_distribution_top.parquet")
    .with_columns(max=pl.max_horizontal(pl.col("AI"), pl.col("HUMAN")))
    .top_k(k=2000, by="max")
    .with_columns(diff=pl.col("AI") - pl.col("HUMAN"))
    .sort("diff", descending=True)
)


def write_cloud(mask, freq, path):
    mask = ((np.array(Image.open(mask).convert("L")) < 150) * 255).astype(np.uint8)

    wc = WordCloud(
        mode="RGBA",
        background_color=None,
        mask=mask,
        colormap="Blues",
        # color_func=color_func,
        max_words=1000
    )

    wc.generate_from_frequencies(
        dict(freq.select("stem", "diff").rows())
    )

    with open(path, "w") as f:
        f.write(wc.to_svg(embed_font=True, embed_image=False))


write_cloud(
    "res/arbeitslos.png", df.top_k(k=1000, by="diff", reverse=True),
    DATA_PATH / "graph_data/word_distribution/human.svg"
)
write_cloud(
    "res/wordcloud_template.png", df.top_k(k=1000, by="diff", reverse=False),
    DATA_PATH / "graph_data/word_distribution/ai.svg"
)

### How much do Al domains cite other Al domains and how do non-Al domains compare?

In [ ]:
stat_schema: dict[str, DataType] = pl.read_parquet_schema(DATA_PATH / "domain_stats.parquet")

df: pl.DataFrame = (
    pl
    .scan_parquet(DATA_PATH / "domain_outflow.parquet")
    .filter(pl.col("from") != pl.col("to"))
    .join(pl.scan_parquet(DATA_PATH / "domain_stats.parquet"), left_on="from", right_on="host")
    .join(pl.scan_parquet(DATA_PATH / "domain_stats.parquet"), left_on="to", right_on="host", suffix="_to")
    .unnest("topic")
    .with_columns(
        pl.col("topic_to")
        .struct
        .rename_fields([f.name + "_to" for f in stat_schema['topic'].fields])
    )
    .unnest("topic_to")
    .collect()
)


def graph_df(x_key: str, y_key: str) -> pl.DataFrame:
    y_key = y_key + "_to"

    return (
        df
        .select(x_key, y_key, "occurrences")
        .group_by(x_key, y_key)
        .agg(pl.col("occurrences").sum())
        .rename({ x_key: "x", y_key: "y" })
    )


def write_frame(x_key: str, y_key: str, nbin: int, file: Path):
    frame = graph_df(x_key, y_key)

    hist = np.histogram2d(
        frame["x"].to_numpy(),
        frame["y"].to_numpy(),
        weights=frame["occurrences"].to_numpy(),
        bins=nbin,
    )

    with open(file, 'w', newline='') as csvfile:
        csvfile.write("x,y,weight\n")
        for idx_x, x in enumerate(hist[1][:-1]):
            for idx_y, y in enumerate(hist[2][:-1]):
                csvfile.write(f"{x},{y},{hist[0][idx_y][idx_x]}\n")


folder = Path(DATA_PATH / "graph_data/network_flow")
folder.mkdir(parents=True, exist_ok=True)

keys = ['ai', 'ai_topic', 'BLOG', 'NEWS', 'WIKI', 'SHOP']

combinations = list(product(keys, keys))

for x_key, y_key in tqdm(combinations):
    write_frame(x_key, y_key, 4, folder / f"x-{x_key}-y-{y_key}.csv")

### How much do Al generated news articles refer to specific categories of sites compared to non-Al articles?

In [ ]:
topic_cols = ['BLOG', 'NEWS', 'WIKI', 'SHOP']


def calculate_flow(filter: pl.IntoExprColumn) -> dict[str, float]:
    return dict(
        list(
            map(
                lambda x: (x[0], x[1][0]),
                pl.scan_parquet(DATA_PATH / "domain_outflow.parquet")
                .filter(pl.col("from") != pl.col("to"))
                .join(pl.scan_parquet(DATA_PATH / "domain_stats.parquet"), left_on="from", right_on="host")
                .join(
                    pl.scan_parquet(DATA_PATH / "domain_stats.parquet"), left_on="to", right_on="host",
                    suffix="_to"
                )
                .filter(filter)
                .select("from", "to", "occurrences", "topic_to")
                .unnest("topic_to")
                .with_columns(
                    max=pl.max_horizontal(*topic_cols),
                )
                .with_columns(
                    *[
                        pl.when(pl.col("max") > 0)
                        .then((pl.col(c) == pl.col("max")))
                        .otherwise(0)
                        .cast(pl.Int8).alias(c)
                        for c in topic_cols
                    ],
                )
                .with_columns(
                    *[
                        pl.when(pl.col("max") > 0)
                        .then(
                            pl.col(c) / pl.sum_horizontal(*topic_cols)
                        ).otherwise(pl.col(c))
                        for c in topic_cols
                    ],
                )
                .with_columns(
                    *[
                        pl.col(c) * pl.col("occurrences")
                        for c in topic_cols
                    ],
                )
                .select(*topic_cols)
                .sum()
                .collect()
                .to_dict()
                .items()
            )
        )
    )


def write_flow(filter: pl.IntoExprColumn, file: Path):
    flow = calculate_flow(filter)

    with open(file, 'w', newline='') as csvfile:
        csvfile.write("topic,outflow\n")
        for topic, flow in flow.items():
            csvfile.write(f"{topic},{flow}\n")


folder = Path(DATA_PATH / "graph_data/topic_references/")

write_flow(
    (pl.col("ai") > 0.7) & (pl.col("topic").struct.field("NEWS") > 0.7),
    folder / "from_ai_news.csv"
)
write_flow(
    (pl.col("ai") < 0.3) & (pl.col("topic").struct.field("NEWS") > 0.7),
    folder / "from_human_news.csv"
)
write_flow(
    (pl.col("ai_topic") > 0.7) & (pl.col("topic").struct.field("NEWS") > 0.7),
    folder / "from_ai_topic_news.csv"
)
write_flow(
    (pl.col("ai_topic") < 0.3) & (pl.col("topic").struct.field("NEWS") > 0.7),
    folder / "from_non_ai_topic_news.csv"
)

### What words most strongly distinguish Al topic pages from non-Al topic pages?

In [ ]:
df = (
    pl
    .scan_parquet(DATA_PATH / "word_distribution.parquet")
    .group_by("ai_topic", "stem")
    .agg(pl.col("occurrence").sum(), pl.col("p").sum())
    .pivot("ai_topic", index="stem", values="p", on_columns=["NOT_AI", "AI"])
    .fill_null(0)
    .collect()
)

threshold = {
    col: df[col].sort(descending=True)[9999]
    for col in ["NOT_AI", "AI"]
}

df = (
    df
    .filter(
        (pl.col("NOT_AI") >= threshold["NOT_AI"])
        | (pl.col("AI") >= threshold["AI"])
    )
    .with_columns(max=pl.max_horizontal(pl.col("AI"), pl.col("NOT_AI")))
    .top_k(k=5000, by="max")
    .drop("max")
    .with_columns(diff=pl.col("AI") - pl.col("NOT_AI"))
    .filter(~pl.col("stem").str.contains_any([",", "=", ":", "("]))
)

(
    df
    .top_k(k=50, by="diff", reverse=False)
    .write_csv("/mnt/Fast2T/data/graph_data/ai_busswords/ai_topic_buzzwords.csv")
)
(
    df
    .top_k(k=50, by="diff", reverse=True)
    .write_csv("/mnt/Fast2T/data/graph_data/ai_busswords/not_ai_topic_words.csv")
)

### What type of sites use what amount of Al buzzwords?

In [ ]:
(
    pl
    .read_parquet(DATA_PATH / "domain_stats.parquet")
    .unnest("topic")
    .unpivot(
        on=['BLOG', 'NEWS', 'WIKI', 'SHOP'],
        index="ai_topic",
        value_name="topic_score",
        variable_name="topic"
    )
    .filter(pl.col("topic_score") > 0.25)
    .group_by("topic")
    .agg(
        pl.col("ai_topic").mean().alias("mean_buzzword"),
        pl.col("topic_score").mean().alias("mean_topic")
    )
    .write_csv(DATA_PATH / "graph_data/ai_busswords/data.csv")
)

### How much do wikis get cited over other sources?

In [ ]:
(
    pl
    .scan_parquet(DATA_PATH / "domain_outflow.parquet")
    .filter(pl.col("from") != pl.col("to"))
    .join(pl.scan_parquet(DATA_PATH / "domain_stats.parquet"), left_on="from", right_on="host")
    .join(
        pl.scan_parquet(DATA_PATH / "domain_stats.parquet"), left_on="to", right_on="host",
        suffix="_to"
    )
    .unnest("topic")
    .with_columns(
        max_score=pl.max_horizontal("NEWS", "BLOG", "WIKI", "SHOP")
    )
    .with_columns(
        pl
        .when(pl.col("max_score") == 0.0).then(pl.lit("UNKNOWN"))
        .when(pl.col("max_score") == pl.col("NEWS")).then(pl.lit("NEWS"))
        .when(pl.col("max_score") == pl.col("BLOG")).then(pl.lit("BLOG"))
        .when(pl.col("max_score") == pl.col("WIKI")).then(pl.lit("WIKI"))
        .when(pl.col("max_score") == pl.col("SHOP")).then(pl.lit("SHOP"))
        .cast(pl.Enum(["NEWS", "SHOP", "WIKI", "BLOG", "UNKNOWN"]))
        .alias("topic")
    )
    .select("from", "to", "topic", "topic_to")
    .unnest("topic_to")
    .with_columns(
        max_score=pl.max_horizontal("NEWS", "BLOG", "WIKI", "SHOP")
    )
    .with_columns(
        pl
        .when(pl.col("max_score") == 0.0).then(pl.lit("UNKNOWN"))
        .when(pl.col("max_score") == pl.col("NEWS")).then(pl.lit("NEWS"))
        .when(pl.col("max_score") == pl.col("BLOG")).then(pl.lit("BLOG"))
        .when(pl.col("max_score") == pl.col("WIKI")).then(pl.lit("WIKI"))
        .when(pl.col("max_score") == pl.col("SHOP")).then(pl.lit("SHOP"))
        .cast(pl.Enum(["NEWS", "SHOP", "WIKI", "BLOG", "UNKNOWN"]))
        .alias("topic_to")
    )
    .select("topic", "topic_to")
    .group_by("topic", "topic_to")
    .len()
    .filter(pl.col("topic") != "UNKNOWN")
    .filter(pl.col("topic_to") != "UNKNOWN")
    .rename({ "topic": "from", "topic_to": "to" })
    .sink_csv(DATA_PATH / "graph_data/sankey_wiki_refs.csv")
)

### How is the composition in terms of Al generation, Al topic, and site type?

In [ ]:
url_df = (
    pl
    .scan_parquet(DATA_PATH / "stage_6/*.parquet")
    .select("host", "url", "is_ai", "ai_topic", "topic")
    .with_columns(
        pl.when(pl.col("topic") == "NEWSAPI")
        .then(pl.lit("NEWS"))
        .otherwise(pl.col("topic"))
        .cast(pl.Enum(["NEWS", "SHOP", "WIKI", "BLOG", "UNKNOWN"]))
        .alias("topic")
    )
    .group_by("is_ai", "ai_topic", "topic")
    .agg(pl.col("url").len())
    .collect()
)

host_df = (
    pl
    .scan_parquet(DATA_PATH / "domain_stats.parquet")
    .drop("samples")
    .with_columns(
        pl
        .when((pl.col("ai") > pl.col("human")) & (pl.col("ai") > 0.5)).then(pl.lit("AI"))
        .when((pl.col("ai") < pl.col("human")) & (pl.col("human") > 0.5)).then(pl.lit("HUMAN"))
        .otherwise(pl.lit("UNKNOWN"))
        .cast(pl.Enum(["AI", "UNKNOWN", "HUMAN"]))
        .alias("is_ai")
    )
    .drop("ai", "human")
    .with_columns(
        pl.when(pl.col("ai_topic") > 0.5).then(pl.lit("AI"))
        .otherwise(pl.lit("NOT_AI"))
        .cast(pl.Enum(["AI", "NOT_AI"]))
        .alias("ai_topic")
    )
    .unnest("topic")
    .with_columns(
        max_score=pl.max_horizontal("NEWS", "BLOG", "WIKI", "SHOP")
    )
    .with_columns(
        pl
        .when(pl.col("max_score") == 0.0).then(pl.lit("UNKNOWN"))
        .when(pl.col("max_score") == pl.col("NEWS")).then(pl.lit("NEWS"))
        .when(pl.col("max_score") == pl.col("BLOG")).then(pl.lit("BLOG"))
        .when(pl.col("max_score") == pl.col("WIKI")).then(pl.lit("WIKI"))
        .when(pl.col("max_score") == pl.col("SHOP")).then(pl.lit("SHOP"))
        .cast(pl.Enum(["NEWS", "SHOP", "WIKI", "BLOG", "UNKNOWN"]))
        .alias("topic")
    )
    .drop("NEWS", "BLOG", "WIKI", "SHOP", "max_score")
    .group_by("is_ai", "ai_topic", "topic")
    .agg(pl.col("host").len())
    .collect()
)

(
    url_df
    .join(host_df, on=["is_ai", "ai_topic", "topic"], how="left")
    .write_csv(DATA_PATH / "graph_data/tree/data.csv")
)